In [ ]:
from sage.all import *
import numpy as np
np.set_printoptions(precision=15)

In [ ]:
approx = ComplexField(32)  # precision 


class GRSBase:
    """GRS subcode for CP code. 
    Note: This works for non-prime fields, unlike the CP code subclass. 
    Parameters:
        length - length of the code; this is equal to q - 1, where q is the 
            size of the base field 
        k - degree of the CP message polynomial (paper's k).
            The associated GRS code GRS(F(k-1,q)) has dimension k.
        chi_ind (optional, 1 by default) - index of the character function, 
            only used in the CP code subclass
    """
    def __init__(self, length, k, chi_ind=1):
        # parameters
        self.length = length
        self.k = k
        # finite field
        self.field_size = length + 1
        F_q.<w> = GF(self.field_size, modulus="primitive")
        self.field = F_q
        self.field_char = self.field.characteristic()
        self.field_units = [w^i for i in range(length)]
        self.chi_ind = chi_ind % self.field_char
        # polynomial ring
        F_q_x.<x> = self.field[]
        self.poly_ring = F_q_x
        # GRS code: GRS(F(k-1, q)) has dimension k
        self.grs = codes.GeneralizedReedSolomonCode(
            self.field_units, 
            self.k, 
            self.field_units
        )
        self.grs_decoder = codes.decoders.GRSBerlekampWelchDecoder
        self.grs_list_decoder = codes.decoders.GRSGuruswamiSudanDecoder
        # pre-compute dimension and minimum distance for __str__ and __eq__
        self._dim = self.dimension()
        self._dmin = self.minimum_distance()

    def __str__(self):
        return (f"({self.length}, {self._dim}, {self._dmin}) CP code with "
                f"character chi_{self.chi_ind} associated with {self.grs}")

    def __repr__(self):
        return self.__str__()

    def __eq__(self, other):
        if isinstance(other, GRSBase):
            return (self._dim == other._dim and self._dmin == other._dmin)
        return NotImplemented

    def polynomial_ring(self):
        """Return the polynomial ring of self."""
        return self.poly_ring

    def __grs_generator_matrix(self):
        """Return a generator matrix of the GRS subcode of self.

        The GRS code encodes g in F(k-1, q) as (alpha_i * g(alpha_i)).
        We need g = f/X where f in F_p(k, q), so g_i = f_{i+1}.
        Forced zeros in f: indices 0, p, 2p, ...
        Forced zeros in g: indices where (i+1) % p == 0, i.e., i = p-1, 2p-1, ...
        """
        mat = copy(self.grs.generator_matrix())
        p = self.field_char
        for i in range(self.k):
            if (i + 1) % p == 0:
                mat[i,:] = 0
        # print(mat)
        return mat

    def grs_subcode(self):
        """Return the GRS subcode of self.grs."""
        # code = LinearCode(self.__grs_generator_matrix())
        # print(code)
        # return code
        return LinearCode(self.__grs_generator_matrix())

    def dimension(self):
        return self.grs_subcode().dimension()

    def minimum_distance(self):
        return self.grs_subcode().minimum_distance()

    def covering_radius(self):
        return self.grs_subcode().covering_radius()


class CharacterPolynomialCode(GRSBase):
    def __init__(self, length, k, chi_ind=1):
        if not is_prime(length + 1):
            raise ValueError("This implementation only works for prime fields")
        super().__init__(length, k, chi_ind)
        # pre-compute character function and its inverse
        self.__chi_mem = dict()
        self.__phi_mem = dict()
        for a in self.field:
            self.__chi_mem[a] = self.__chi(a)
            self.__phi_mem[self.__chi_mem[a]] = a

    def __contains__(self, c):
        """
        If c is a polynomial, return True iff c is in the message space of self. 
        If c is a vector, return True iff c is a valid codeword of self. 
        """
        if isinstance(
            c, 
            sage.rings.polynomial.polynomial_zmod_flint.Polynomial_zmod_flint
        ):
            for i, coeff in enumerate(c.list()):
                if i % self.field_char == 0 and coeff != 0:
                    return False
            return c in self.poly_ring and c.degree() < self.k + 1
        if isinstance(
            c, 
            sage.modules.free_module_element.FreeModuleElement_generic_dense
        ):
            try:
                f = self.decode(c) 
                return self.encode(f) == c
            except sage.coding.decoder.DecodingError:
                return False
        return False

    def dimension(self):
        """Paper's equation (3): dimension = k - floor(k/p)"""
        return self.k - self.k // self.field_char

    def minimum_distance(self):
        """GRS(F(k-1,q)) has d = n - k + 1"""
        return self.length - self.k + 1

    def covering_radius(self):
        return self.minimum_distance() - 1

    def decoding_radius(self):
        return self.grs_decoder(self.grs).decoding_radius()

    def list_decoding_radius(self):
        return self.grs_list_decoder.guruswami_sudan_decoding_radius(self.grs)[0]

    def convert_polynomial(self, f):
        """Convert f to a polynomial in the message space of self by setting 
        every p-th coefficient equal to 0, where p = self.field_char.
        """
        if not (f in self.poly_ring and f.degree() < self.k + 1):
            raise ValueError(f"{f} is not an element of {self.poly_ring} "
                             f"of degree less than {self.k + 1}")
        coeffs = f.list()
        for i in range(len(coeffs)):
            if i % self.field_char == 0:
                coeffs[i] = 0
        return self.poly_ring(coeffs)

    def __chi(self, a):
        """Compute the additive character of self.field corresponding to a."""
        if not a in self.field:
            raise ValueError(f"{a} is not an element of {self.field}")
        ja = self.chi_ind * a
        return exp(2*pi*I*lift(ja.trace())/self.field_char)

    def __encode_raw(self, f):
        """Encode f directly, i.e. without using the built-in GRS encoder."""
        cp_rs_cw = vector(
            self.field, 
            [a * f(a) for a in self.field_units]
        )
        cp_vec = [self.__chi_mem[c] for c in cp_rs_cw]
        return vector(approx, cp_vec)

    def encode(self, f, debug=False):
        if not f in self:
            raise ValueError(f"{f} is not an element of the message space "
                             f"of {self}")
        g = f // self.poly_ring.gen()
        cp_rs_cw = self.grs.encoder("EvaluationPolynomial").encode(g)
        if debug:
            print(cp_rs_cw)
        cp_vec = [self.__chi_mem[c] for c in cp_rs_cw]
        cp_cw = vector(approx, cp_vec)
        if debug:
            print("Encoder success?", cp_cw == self.__encode_raw(g))
        return cp_cw

    def __nearest_img(self, z):
        """Return the nearest point (in Euclidean distance) to z in chi(F_q)."""
        p = self.field_char
        z = approx(z)
        if z.arg() >= 0:
            ex = floor(self.length*approx(z.arg())/(2*pi) + 1/2)
        else: 
            ex = (p+1)//2 + floor(self.length*approx(pi+z.arg())/(2*pi) + 1/2)
        return exp(2*pi*I/p)^ex

    def __phi(self, z):
        """Return the pre-image of the nearest point to z in chi(F_q)."""
        z_to_unit_circle = self.__nearest_img(z)
        return self.__phi_mem.get(z_to_unit_circle)

    def __pre_process(self, m, debug=False):
        """Return the result of applying phi to each coordinate of m."""
        assert(len(m) == self.length)
        y = [0] * self.length
        for i, mi in enumerate(m):
            y[i] = self.__phi(mi)
        ynew = vector(self.field, y)
        if debug:
            print(f"After phi:\n\t{ynew}")
        return ynew

    def decode(self, m, debug=False):
        if debug:
            print(f"Running CP decoder using {self.grs_decoder} on received "
                  f"message {m}")
        ynew = self.__pre_process(m, debug)
        return (self.poly_ring.gen() * 
                self.grs_decoder(self.grs).decode_to_message(ynew))

    def list_decode(self, m, debug=False):
        if debug:
            print(f"Running CP list decoder using {self.grs_list_decoder} on "
                  f"received message {m}")
        ynew = self.__pre_process(m, debug)
        gs_output = self.grs_list_decoder(
            self.grs, 
            tau=self.list_decoding_radius()
        ).decode_to_message(ynew)
        grs_list = [self.poly_ring.gen() * f for f in gs_output]
        return [f for f in grs_list if f in self]

In [ ]:
class codebook:
    def __init__(self):


        self.f_psk = None
        self.O1 = None
        self.codebook_cp = None
        self.code = None

        self.Nt = None
        self.Nr = None

        self.H = None
        self.f_opt = None
        self.f_cp = None
        self.f_dft = None

        self.codebook_dft = None


    def psk_decode(self, f_opt, M):
        eta = np.cos( (2*np.pi)/M ) + 1j * np.sin( (2*np.pi)/M )
        arg = (np.angle(f_opt) * M) / ( 2*np.pi )
        g = round(arg)
        u = np.argsort(g - arg)
        eta_gt = np.array([])
        for i in range(g.shape[0]):
            eta_gt = np.append (eta_gt, np.cos( (2*np.pi * g[i])/M ) + 1j * np.sin( (2*np.pi * g[i] )/M ))
        p = f_opt * eta_gt
        v = np.hstack((np.sum(p), p[u]*(eta-1)))

        best = np.argmax(np.abs(np.cumsum(v)))
        g[u[1:best-1]] = g[u[1:best-1]] + 1

        g = (g-g[0])%M

        return g

    def DFT_codebook(self, oversampling):
        self.O1 = oversampling
        n = np.arange(self.Nt)
        l = np.arange(self.Nt*self.O1)
        codebook_dft = np.exp(1j*2*np.pi*np.outer(n,l)/(self.Nt*self.O1))/ np.sqrt(self.Nt)
        return codebook_dft.T

    def CP_codebook(self, code):
        F = GF(Nt + 1)
        R.<x> = PolynomialRing(F)

        # CP code condition
        assert chi_i <= Nt

        self.code = code

        # Enumerate all polynomials of degree <= k
        polynomials = R.polynomials(max_degree=deg)

        codebook_cp = []
        poly_c = set()

        for poly in polynomials:
            poly_c.add(self.code.convert_polynomial(poly))

        for poly in poly_c:
            codebook_cp.append(self.code.encode(poly))

        codebook_cp = np.array(codebook_cp)
        return codebook_cp

    def PSK_codeword(self, M, f_opt):
        psk_vec = np.zeros_like(f_opt)
        psk_ind = self.psk_decode(f_opt, M)
        for k in range(psk_vec.shape[0]):
            psk_vec[k] = (1/ np.sqrt(Nt) ) * ( np.cos((2*np.pi/M)*psk_ind[k]) + 1j*np.sin((2*np.pi/M)*psk_ind[k]) )
        return psk_vec


    def channel(self, ch_type):
        H_mu, H_var = 0, sqrt(1/2)
        if ch_type == "Rayleigh":
            H = ((np.random.normal(H_mu, H_var, (self.Nr, self.Nt))) + 1j * (np.random.normal(H_mu, H_var, (self.Nr ,self.Nt))))        
        elif ch_type == "Rician":
            K_factor = 0.25
            los = np.sqrt(K_factor/(K_factor+1))
            nlos = ((np.random.normal(H_mu, H_var, (self.Nr, self.Nt))) + 1j * (np.random.normal(H_mu, H_var, (self.Nr ,self.Nt))))
            H = los + nlos
        else:
            raise Exception('Invalid channel type')
        return H

    def ideal_codeword(self, Nr, H):
        U, Sigma, Vh = np.linalg.svd(H, full_matrices=False)

        # Find the ideal infinite precision vector
        if Nr == 1:
            f_opt = np.conjugate(np.transpose(Vh))
            f_opt = f_opt.reshape(-1)
        elif Nr > 1:
            f_opt = np.conjugate(np.transpose(Vh[0,:]))
            f_opt = f_opt.reshape(-1)
        else:
            raise ValueError("Invalid receive number of antennas")
        return f_opt

    def codeword_search(self, codebook, f_opt):
        # if corr == "Uncorrelated" and r_tx is None:
        #     codebook = codebook/np.linalg.norm(codebook, axis=1, keepdims=True)
        # elif corr == "Correlated" and r_tx.size>1 and r_tx is not None:
        #     """
        #     # codebook = (np.conjugate(r_tx.T) @ codebook.T).T
        #     
        #      Rotating codebook makes gain no longer equal gains but more than EGT gain due to R_tx rotation for larger rho_tx values. 
        #      Baseline will be MRT not EGT.
        #      So better to avoid rotation. In some sense TX has no idea about R_tx and yet codebook is superior.
        #      
        #     """
        #     codebook = codebook/np.linalg.norm(codebook, axis=1, keepdims=True)
        # else:
        #     raise ValueError("Invalid correlation type")
        codebook = codebook/np.linalg.norm(codebook, axis=1, keepdims=True)
        # print(codebook.shape, f_opt.shape)
        max_index = np.argmax(np.abs(codebook.conj() @ f_opt)**2)
        f = codebook[max_index]
        return f

    def MRC_gain(self, f_opt, H):
        gain = ((np.linalg.norm(H @ f_opt))**2)
        return gain

    def EGT_gain(self, Nr, H, Nt):
        if Nr == 1:
            gain = ((np.sum(np.abs(H)))**2/Nt)
        else:
            raise Exception("Exact EGT gain doesn't exist for Nr>1")
        return gain

    def CP_gain(self, H, f):
        gain = (np.linalg.norm(H @ f)**2)
        return gain

    def PSK_gain(self, H, f):
        # print(H.shape,f.shape)
        PSK_gain = (np.linalg.norm(H @ f)**2)
        return PSK_gain

    def DFT_gain(self, H, f):
        DFT_gain = (np.linalg.norm(H @ f)**2)
        return DFT_gain







In [ ]:
# from types import SimpleNamespace
# 
# class AutoStruct(SimpleNamespace):
#     def __getattr__(self, name):
#         obj = AutoStruct()
#         setattr(self, name, obj)
#         return obj


class Channel:
    def __init__(self):
        np.random.seed(42)
        self.H_mu = None    
        self.H_var = None
        
    # def __getattr__(self, name):
    #     obj = AutoStruct()
    #     setattr(self, name, obj)
    #     return obj
        
    def Rayleigh(self, Nr, Nt, corr, rho_tx, rho_rx, r_tx, r_rx):
        if corr == "Correlated" and rho_tx is not None and rho_rx is not None and r_tx is not None and r_rx is not None:
            C = np.linalg.cholesky(r_tx)
            if Nr == 1:
                if rho_rx == 1:
                    H_rayleigh = ((np.random.normal(self.H_mu, self.H_var, (Nr, Nt))) + 1j * (np.random.normal(self.H_mu, self.H_var, (Nr ,Nt))))
                    H_corr_rayleigh = H_rayleigh @ C.T
                    return H_corr_rayleigh
                else:
                    raise ValueError("rho_rx must be 1 for Nr=1")
            else:
                r_rx = np.array([[rho_tx**abs(i-j) for j in range(Nr)] for i in range(Nr)])
                D = np.linalg.cholesky(r_rx)
                H_rayleigh = ((np.random.normal(self.H_mu, self.H_var, (Nr, Nt))) + 1j * (np.random.normal(self.H_mu, self.H_var, (Nr ,Nt))))
                H_corr_rayleigh = D @ H_rayleigh @ C.T
                return H_corr_rayleigh
        elif corr == "Uncorrelated" and rho_tx is None and rho_rx is None and r_tx is None and r_rx is None:
            H_rayleigh = ((np.random.normal(self.H_mu, self.H_var, (Nr, Nt))) + 1j * (np.random.normal(self.H_mu, self.H_var, (Nr ,Nt))))
            return H_rayleigh
        else:
            raise Exception("Invalid Rayleigh channel")
    
    def Rician(self, Nr, Nt, k_factor, corr, rho_tx, rho_rx, r_tx, r_rx):      
        if corr == "Correlated" and rho_tx is not None and rho_rx is not None and r_tx is not None and r_rx is not None:
            C = np.linalg.cholesky(r_tx)
            if Nr == 1:
                if rho_rx == 1:
                    los = np.sqrt(k_factor/(k_factor+1))
                    nlos = np.sqrt(1/(k_factor+1))* ((np.random.normal(self.H_mu, self.H_var, (Nr, Nt))) + 1j * (np.random.normal(self.H_mu, self.H_var, (Nr ,Nt))))
                    H_rician = los + nlos
                    H_corr_rician = H_rician @ C.T
                    return H_corr_rician
                else:
                    raise ValueError("rho_rx must be 1 for Nr=1")
            else:
                r_rx = np.array([[rho_tx**abs(i-j) for j in range(Nr)] for i in range(Nr)])
                D = np.linalg.cholesky(r_rx)
                los = np.sqrt(k_factor/(k_factor+1))
                nlos = np.sqrt(1/(k_factor+1))* ((np.random.normal(self.H_mu, self.H_var, (Nr, Nt))) + 1j * (np.random.normal(self.H_mu, self.H_var, (Nr ,Nt))))
                H_rician = los + nlos
                H_corr_rician = D @ H_rician @ C.T
                return H_corr_rician
        elif corr == "Uncorrelated" and rho_tx is None and rho_rx is None and r_tx is None and r_rx is None:
            los = np.sqrt(k_factor/(k_factor+1))
            nlos = np.sqrt(1/(k_factor+1))* ((np.random.normal(self.H_mu, self.H_var, (Nr, Nt))) + 1j * (np.random.normal(self.H_mu, self.H_var, (Nr ,Nt))))
            H_rician = los + nlos
            return H_rician
        else:
            raise Exception("Invalid Rician channel")
    
    # def gen_cdl(self, model, Nr, Nt, d_lambda=0.5):
    #     """
    #     Generate one Nr x Nt narrowband channel matrix from 3GPP CDL.
    #     
    #     H = gen_cdl("A", Nt=4)          # 1x4 MISO
    #     H = gen_cdl("A", Nt=4, Nr=2)    # 2x4 MIMO
    #     """
    #     profiles = {
    #         "A": {"p": [-13.4,0,-2.2,-4,-6,-8.2,-9.9,-10.5,-7.5,-15.9,-6.6,-16.7,-12.4,-15.2,-10.8,-11.3,-12.7,-16.2,-18.3,-18.9,-16.6,-19.9,-29.7],
    #               "a": [-178.1, -4.2, -4.2, -4.2, 90.2, 90.2, 90.2,
    #   121.5, -81.7, 158.4, -83.0, 134.8, -153.0,
    #   -172.0, -129.9, -136.0, 165.4, 148.4, 132.7,
    #   -118.6, -154.1, 126.5, -56.2], "K": None},
    #         "B": {"p": [0,-2.2,-4,-3.2,-9.8,-1.2,-3.4,-5.2,-7.6,-3,-8.9,-9,-4.8,-5.7,-7.5,-1.9,-7.6,-12.2,-9.4,-11.4,-14.9,-9.2,-11.7],
    #               "a": [9.3,9.3,9.3,-34.1,-34.1,-65.4,-65.4,-65.4,-65.4,-101.2,-101.2,-101.2,60.5,60.5,60.5,-37.4,-37.4,-37.4,-122.4,-122.4,-122.4,80.9,80.9], "K": None},
    #         "C": {"p": [-4.4,-1.2,-3.5,-5.2,-2.5,0,-2.2,-3.9,-7.4,-7.1,-10.7,-11.1,-5.1,-6.8,-8.7,-13.2,-13.9,-13.9,-15.8,-17.1,-16,-15.7,-21.6,-22.8],
    #               "a": [-46.6,-46.6,-46.6,-101.8,-101.8,17.2,17.2,17.2,17.2,-74.3,-74.3,-74.3,127.7,127.7,127.7,-119.6,-119.6,-119.6,-9.1,-9.1,-9.1,150.8,150.8,150.8], "K": None},
    #         "D": {"p": [-0.2,-13.5,-18.8,-21,-22.8,-17.9,-20.1,-21.9,-22.9,-27.8,-23.6,-24.8,-30],
    #               "a": [0,0,89.2,89.2,89.2,-89.2,-89.2,-89.2,-89.2,13,13,-13,-13], "K": 13.3},
    #         "E": {"p": [-0.03,-22.03,-15.8,-18.1,-19.8,-22.9,-22.4,-18.6,-20.8,-22.6,-22.3,-25.6,-20.2,-29.8,-29.2],
    #               "a": [0,0,57.5,57.5,57.5,-20.1,-20.1,47.4,47.4,47.4,110.4,110.4,-48.1,-48.1,-48.1], "K": 22.4},
    #     }
    #     pr = profiles[model]
    #     P = 10.0**(np.array(pr["p"])/10.0); P /= P.sum()
    #     phi = np.deg2rad(pr["a"])
    #     N_cl = len(P)
    # 
    #     # Steering matrix: (N_cl, Nt)
    #     A = np.exp(1j*2*np.pi*d_lambda*np.outer(np.sin(phi), np.arange(Nt)))
    # 
    #     # Fading: (Nr, N_cl) — one independent fading coeff per Rx antenna per cluster
    #     G = (np.random.randn(Nr, N_cl) + 1j*np.random.randn(Nr, N_cl)) / np.sqrt(2)
    #     G *= np.sqrt(P)
    # 
    #     # LOS handling
    #     if pr["K"] is not None:
    #         K = 10.0**(pr["K"]/10.0)
    #         G[:, 0] *= np.sqrt(1.0/(K+1.0))
    #         H = G @ A + np.sqrt(P[0]*K/(K+1.0)) * A[0]  # (Nr, Nt)
    #     else:
    #         H = G @ A  # (Nr, Nt)
    # 
    #     return H
    
    def gen_cdl(self, model, Nr, Nt, d_lambda=0.5):
        """
        Generate one Nr x Nt narrowband channel matrix from 3GPP CDL.
        TR 38.901 Tables 7.7.1-1 through 7.7.1-5.
    
        H = gen_cdl("A", Nt=4)          # 1x4 MISO
        H = gen_cdl("A", Nt=4, Nr=2)    # 2x4 MIMO
        """
        profiles = {
            # ── CDL-A  (Table 7.7.1-1, NLOS, 23 clusters) ── VERIFIED ──
            "A": {
                "p": [-13.4, 0, -2.2, -4, -6, -8.2, -9.9, -10.5, -7.5, -15.9,
                      -6.6, -16.7, -12.4, -15.2, -10.8, -11.3, -12.7, -16.2,
                      -18.3, -18.9, -16.6, -19.9, -29.7],
                "aod": [-178.1, -4.2, -4.2, -4.2, 90.2, 90.2, 90.2,
                        121.5, -81.7, 158.4, -83.0, 134.8, -153.0,
                        -172.0, -129.9, -136.0, 165.4, 148.4, 132.7,
                        -118.6, -154.1, 126.5, -56.2],
                "aoa": [51.3, -152.7, -152.7, -152.7, 76.6, 76.6, 76.6,
                        -1.8, -41.9, 94.2, 51.9, -115.9, 26.6,
                        76.6, -7.0, -23.0, -47.2, 110.4, 144.5,
                        155.3, 102.0, -151.8, 55.2],
                "K": None,
            },
    
            # ── CDL-B  (Table 7.7.1-2, NLOS, 23 clusters) ── NEEDS VERIFICATION ──
            # TODO: replace aod/aoa with MATLAB dump: nrCDLChannel 'CDL-B' → info()
            "B": {
                "p": [0, -2.2, -4, -3.2, -9.8, -1.2, -3.4, -5.2, -7.6, -3,
                      -8.9, -9, -4.8, -5.7, -7.5, -1.9, -7.6, -12.2,
                      -9.4, -11.4, -14.9, -9.2, -11.7],
                "aod": [9.3, 9.3, 9.3, -34.1, -34.1, -65.4, -65.4, -65.4, -65.4,
                        -101.2, -101.2, -101.2, 60.5, 60.5, 60.5, -37.4, -37.4,
                        -37.4, -122.4, -122.4, -122.4, 80.9, 80.9],  # UNVERIFIED
                "aoa": [0]*23,  # PLACEHOLDER — fill from MATLAB
                "K": None,
            },
    
            # ── CDL-C  (Table 7.7.1-3, NLOS, 24 clusters) ── NEEDS VERIFICATION ──
            # TODO: replace aod/aoa with MATLAB dump: nrCDLChannel 'CDL-C' → info()
            "C": {
                "p": [-4.4, -1.2, -3.5, -5.2, -2.5, 0, -2.2, -3.9, -7.4, -7.1,
                      -10.7, -11.1, -5.1, -6.8, -8.7, -13.2, -13.9, -13.9,
                      -15.8, -17.1, -16, -15.7, -21.6, -22.8],
                "aod": [-46.6, -46.6, -46.6, -101.8, -101.8, 17.2, 17.2, 17.2, 17.2,
                        -74.3, -74.3, -74.3, 127.7, 127.7, 127.7, -119.6, -119.6,
                        -119.6, -9.1, -9.1, -9.1, 150.8, 150.8, 150.8],  # UNVERIFIED
                "aoa": [0]*24,  # PLACEHOLDER — fill from MATLAB
                "K": None,
            },
    
            # ── CDL-D  (Table 7.7.1-4, LOS, 13 clusters) ── VERIFIED ──
            # idx 0 = specular (LOS), idx 1 = Laplacian part of cluster 1
            # idx 2..13 = clusters 2–13
            "D": {
                "p": [-0.2, -13.5, -18.8, -21, -22.8, -17.9, -20.1, -21.9,
                      -22.9, -27.8, -23.6, -24.8, -30.0, -27.7],
                "aod": [0, 0, 89.2, 89.2, 89.2, 13, 13, 13,
                        34.6, -64.5, -32.9, 52.6, -132.1, 77.2],
                "aoa": [-180, -180, 89.2, 89.2, 89.2, 163, 163, 163,
                        -137, 74.5, 127.7, -119.6, -9.1, -83.8],
                "K": 13.3,
            },
    
            # ── CDL-E  (Table 7.7.1-5, LOS, 15 clusters) ── NEEDS VERIFICATION ──
            # TODO: replace aod/aoa with MATLAB dump: nrCDLChannel 'CDL-E' → info()
            "E": {
                "p": [-0.03, -22.03, -15.8, -18.1, -19.8, -22.9, -22.4, -18.6,
                      -20.8, -22.6, -22.3, -25.6, -20.2, -29.8, -29.2],
                "aod": [0, 0, 57.5, 57.5, 57.5, -20.1, -20.1, 47.4, 47.4, 47.4,
                        110.4, 110.4, -48.1, -48.1, -48.1],  # UNVERIFIED
                "aoa": [0]*15,  # PLACEHOLDER — fill from MATLAB
                "K": 22.4,
            },
        }
    
        pr = profiles[model]
        P = 10.0**(np.array(pr["p"]) / 10.0)
        P /= P.sum()
        phi_tx = np.deg2rad(pr["aod"])
        phi_rx = np.deg2rad(pr["aoa"])
        N_cl = len(P)
    
        # One scalar fading coefficient per cluster
        g = (np.random.randn(N_cl) + 1j * np.random.randn(N_cl)) / np.sqrt(2)
    
        # LOS: kill fading on cluster 0 (it's deterministic)
        if pr["K"] is not None:
            g[0] = 0.0
    
        # Build H as sum of rank-1 outer products
        H = np.zeros((Nr, Nt), dtype=complex)
        for n in range(N_cl):
            a_tx = np.exp(1j * 2*np.pi * d_lambda * np.sin(phi_tx[n]) * np.arange(Nt))
            a_rx = np.exp(1j * 2*np.pi * d_lambda * np.sin(phi_rx[n]) * np.arange(Nr))
            H += np.sqrt(P[n]) * g[n] * np.outer(a_rx, a_tx)
    
        # Add deterministic LOS component (P[0] IS the LOS power from the table)
        if pr["K"] is not None:
            a_tx0 = np.exp(1j * 2*np.pi * d_lambda * np.sin(phi_tx[0]) * np.arange(Nt))
            a_rx0 = np.exp(1j * 2*np.pi * d_lambda * np.sin(phi_rx[0]) * np.arange(Nr))
            H += np.sqrt(P[0]) * np.outer(a_rx0, a_tx0)
    
        return H
        

In [ ]:
# MIMO parameters
Nt = 10
assert  True == is_prime(Nt+1)
Nr = 1


# Monte Carlo parameters
N_sim = 300
MRC_gain_dB = np.array([])
EGT_gain_dB = np.array([])
CP_gain_dB = np.array([])
PSK_gain_dB = np.array([])
DFT_gain_dB = np.array([])

B_cp = np.array([])
B_psk = np.array([])
B_dft = np.array([])

# CP code parameters
if Nt>6:
    k = np.arange(1, 7)
else:
    k = np.arange(1, Nt+1)
chi_i = 1 

# DFT parameters

if Nt == 4:
    O1_val = np.array([1, 6, 31, 156])
elif Nt == 6:
    O1_val = np.array([1, 8, 57, 400, 2801, 19608])
elif Nt == 10:
    O1_val = np.array([1, 12, 133, 1464, 16105, 177156, 1948717])
else:
    raise Exception("Nt=p-1 where prime p is 5<=p<=11")

MIMO = codebook()

MIMO.Nt = Nt
MIMO.Nr = Nr

#

H_mu, H_var = 0, sqrt(1/2)
K_factor = 0.1



# ch_type = "Rayleigh"
# # ch_type = "Rician"
# chcorr_type = "Uncorrelated"
# 
# chcorr_type = "Correlated"
# Rho_tx, Rho_rx = 0.75, 1
# R_tx = np.array([[Rho_tx**abs(i-j) for j in range(Nt)] for i in range(Nt)])
# R_rx = np.array([[Rho_tx**abs(i-j) for j in range(Nr)] for i in range(Nr)])


ch_type = "CDL"
cdl_profile = "A"
# cdl_profile = "D"



Ch = Channel()

Ch.H_mu = H_mu
Ch.H_var = H_var

Ch.ch_type = ch_type

if Ch.ch_type == "CDL":
    Ch.cdl_profile = cdl_profile
elif Ch.ch_type == "Rician":
    Ch.chcorr_type = chcorr_type
    Ch.K_factor = K_factor
    if Ch.chcorr_type == "Correlated":
        Ch.Rho_tx, Ch.Rho_rx = Rho_tx, Rho_rx
        Ch.R_tx, Ch.R_rx = R_tx, R_rx
    elif Ch.chcorr_type == "Uncorrelated":
        Ch.Rho_tx, Ch.Rho_rx = None, None
        Ch.R_tx, Ch.R_rx = None, None
elif Ch.ch_type == "Rayleigh":
    Ch.chcorr_type = chcorr_type
    if Ch.chcorr_type == "Correlated":
        Ch.Rho_tx, Ch.Rho_rx = Rho_tx, Rho_rx
        Ch.R_tx, Ch.R_rx = R_tx, R_rx
    elif Ch.chcorr_type == "Uncorrelated":
        Ch.Rho_tx, Ch.Rho_rx = None, None
        Ch.R_tx, Ch.R_rx = None, None
else:
    raise ValueError("Invalid Channel type")

# M_list = np.arange(2,10,2)
M_inc = 2
M_init = 2


rows = []

for deg in k:
    print("degree =", deg)
    MRC_gain = np.array([])    
    EGT_gain = np.array([])
    CP_gain = np.array([])
    PSK_gain = np.array([])
    DFT_gain = np.array([])
    
    
    MIMO.codebook_cp = MIMO.CP_codebook(CharacterPolynomialCode(MIMO.Nt, deg, chi_i))
    print("Codebook generated for degree {}".format(deg), MIMO.codebook_cp.shape)
    B_cp = np.append(B_cp, np.ceil(deg*np.log2(Nt+1)))
    
    MIMO.codebook_dft = MIMO.DFT_codebook(O1_val[deg-1])
    B_dft = np.append(B_dft, np.ceil(np.log2(O1_val[deg-1]*Nt)))
    
    M = M_init + (deg-1)*M_inc
    B_psk = np.append(B_psk, np.ceil((Nt-1)*np.log2(M)))
    
    np.random.seed(42)
    for i in range(N_sim):
        
        
        if ch_type == "Rayleigh":
            H = Ch.Rayleigh(MIMO.Nr, MIMO.Nt, Ch.chcorr_type, Ch.Rho_tx, Ch.Rho_rx, Ch.R_tx, Ch.R_rx)
            # print(H.shape)
        elif ch_type == "Rician":
            H = Ch.Rician(MIMO.Nr, MIMO.Nt, Ch.K_factor, Ch.chcorr_type, Ch.Rho_tx, Ch.Rho_rx, Ch.R_tx, Ch.R_rx)
        elif ch_type == "CDL":
            H = Ch.gen_cdl(Ch.cdl_profile, MIMO.Nr, MIMO.Nt)
            # print(H.shape)
        else:
            raise Exception('Invalid channel type')
        

        MIMO.f_opt = MIMO.ideal_codeword(MIMO.Nr, H)

        MRC_gain = np.append(MRC_gain, MIMO.MRC_gain(MIMO.f_opt, H))
        if MIMO.Nr == 1:
            EGT_gain = np.append(EGT_gain, MIMO.EGT_gain(MIMO.Nr, H, MIMO.Nt))
            

        MIMO.f_cp = MIMO.codeword_search(MIMO.codebook_cp, MIMO.f_opt)
        MIMO.f_psk = MIMO.PSK_codeword(M, MIMO.f_opt)
        MIMO.f_dft = MIMO.codeword_search(MIMO.codebook_dft, MIMO.f_opt)

        CP_gain = np.append(CP_gain, MIMO.CP_gain(H, MIMO.f_cp))
        PSK_gain = np.append(PSK_gain,MIMO.PSK_gain(H, MIMO.f_psk))
        DFT_gain = np.append(DFT_gain, MIMO.DFT_gain(H, MIMO.f_dft))
        
        if i%50 == 0:
            print("mont_compl::",i)
    
    MRC_gain_tot = np.sum(MRC_gain)/N_sim
    if MIMO.Nr == 1:
        EGT_gain_tot = np.sum(EGT_gain)/N_sim
    CP_gain_tot = np.sum(CP_gain)/N_sim
    PSK_gain_tot = np.sum(PSK_gain)/N_sim
    DFT_gain_tot = np.sum(DFT_gain)/N_sim
    
    MRC_gain_dB = np.append(MRC_gain_dB, 10*np.log10(MRC_gain_tot))
    if MIMO.Nr == 1:
        EGT_gain_dB = np.append(EGT_gain_dB,10*np.log10(EGT_gain_tot))
    CP_gain_dB = np.append(CP_gain_dB,10*np.log10(CP_gain_tot))
    PSK_gain_dB = np.append(PSK_gain_dB,10*np.log10(PSK_gain_tot))
    DFT_gain_dB = np.append(DFT_gain_dB,10*np.log10(DFT_gain_tot))
    
    # file saving
    
    if N_sim == 300:
        if MIMO.Nr == 1:
            rows.append([MIMO.Nt, deg, EGT_gain_dB[deg-1], CP_gain_dB[deg-1], B_cp[deg-1], PSK_gain_dB[deg-1], B_psk[deg-1], DFT_gain_dB[deg-1], B_dft[deg-1] ])
            data = np.array(rows)
            header = (
            f"{'NT':<6}{'k':<6}{'EGT':<22}{'CP':<22}{'B_CP':<22}"
            f"{'PSK':<22}{'B_PSK':<22}{'DFT':<22}{'B_DFT':<22}")
            if Ch.ch_type == "Rayleigh" or Ch.ch_type == "Rician":
                np.savetxt(str(MIMO.Nt)+"x"+str(MIMO.Nr)+"_"+str(Ch.ch_type)+str(Ch.chcorr_type)+"_"+str(Ch.Rho_tx)+"_"+str(Ch.Rho_rx), data,
                header=header,
                fmt=["%-6.0f", "%-6.0f", "%-22.15f", "%-22.15f", "%-22.15f",
                 "%-22.15f", "%-22.15f", "%-22.15f", "%-22.15f"],
                delimiter="",
                comments=""
                )
            elif Ch.ch_type == "CDL":
                np.savetxt(str(MIMO.Nt)+"x"+str(MIMO.Nr)+"_"+str(Ch.ch_type)+str(Ch.cdl_profile), data,
                header=header,
                fmt=["%-6.0f", "%-6.0f", "%-22.15f", "%-22.15f", "%-22.15f",
                 "%-22.15f", "%-22.15f", "%-22.15f", "%-22.15f"],
                delimiter="",
                comments=""
                )       
            print("EGT gain: ", EGT_gain_dB)
            print("CP gain: ", CP_gain_dB)
            print("B_CP", B_cp)
            print("PSK gain: ", PSK_gain_dB)
            print("B_PSK", B_psk)
            print("DFT gain: ", DFT_gain_dB)
            print("B_DFT", B_dft)

        else:
            rows.append([MIMO.Nt, deg, CP_gain_dB[deg-1], B_cp[deg-1], PSK_gain_dB[deg-1], B_psk[deg-1], DFT_gain_dB[deg-1], B_dft[deg-1] ])
            data = np.array(rows)
            header = (
            f"{'NT':<6}{'k':<6}{'CP':<22}{'B_CP':<22}"
            f"{'PSK':<22}{'B_PSK':<22}{'DFT':<22}{'B_DFT':<22}")
            if Ch.ch_type == "Rayleigh" or Ch.ch_type == "Rician":
                np.savetxt(str(MIMO.Nt)+"x"+str(MIMO.Nr)+"_"+str(Ch.ch_type)+str(Ch.chcorr_type)+"_"+str(Ch.Rho_tx)+"_"+str(Ch.Rho_rx), data,
                header=header,
                fmt=["%-6.0f", "%-6.0f", "%-22.15f", "%-22.15f",
                 "%-22.15f", "%-22.15f", "%-22.15f", "%-22.15f"],
                delimiter="",
                comments=""
                )
            elif Ch.ch_type == "CDL":
                np.savetxt(str(MIMO.Nt)+"x"+str(MIMO.Nr)+"_"+str(Ch.ch_type)+str(Ch.cdl_profile), data,
                header=header,
                fmt=["%-6.0f", "%-6.0f", "%-22.15f", "%-22.15f",
                 "%-22.15f", "%-22.15f", "%-22.15f", "%-22.15f"],
                delimiter="",
                comments=""
                )   
    print("EGT gain: ", EGT_gain_dB)
    print("CP gain: ", CP_gain_dB)
    print("B_CP", B_cp)
    print("PSK gain: ", PSK_gain_dB)
    print("B_PSK", B_psk)
    print("DFT gain: ", DFT_gain_dB)
    print("B_DFT", B_dft)
    
    

In [ ]:
import matplotlib.pyplot as plt
B_max = np.max([B_cp,B_psk,B_dft])
b = np.arange(1,B_max+1,1)




if Nr == 1:
    plt.title(str(Ch.chcorr_type)+ " " + str(Ch.ch_type) + " fading channel in 1x"+str(Nt)+ " MISO system")
    plt.xticks(b)
    plt.plot(b, EGT_gain_dB[0]*np.ones_like(b), color='black', linestyle='-',  linewidth=2, markersize=7, label='EGT')
    plt.plot(B_cp, CP_gain_dB, color='tab:blue',marker='s', linestyle='--', linewidth=2, markersize=7, label='CP')
    plt.plot(B_psk, PSK_gain_dB, color='tab:green', marker='^', linestyle='-.', linewidth=2, markersize=7, label='PSK')
    plt.plot(B_dft, DFT_gain_dB, color='tab:red',   marker='d', linestyle=':',  linewidth=2, markersize=7, label='DFT')
    plt.xlabel("Feedback bits")
    plt.ylabel("Gain[dB]")
    plt.legend()
elif Nr > 1:
    plt.title(str(Ch.chcorr_type)+ " " + str(Ch.ch_type) + " fading channel in 1x"+str(Nt)+ " MIMO system")
    plt.xticks(b)
    # plt.plot(b, EGT_gain_dB[0]*np.ones_like(b), color='black', linestyle='-',  linewidth=2, markersize=7, label='EGT')
    plt.plot(B_cp, CP_gain_dB, color='tab:blue',marker='s', linestyle='--', linewidth=2, markersize=7, label='CP')
    plt.plot(B_psk, PSK_gain_dB, color='tab:green', marker='^', linestyle='-.', linewidth=2, markersize=7, label='PSK')
    plt.plot(B_dft, DFT_gain_dB, color='tab:red',   marker='d', linestyle=':',  linewidth=2, markersize=7, label='DFT')
    plt.xlabel("Feedback bits")
    plt.ylabel("Gain[dB]")
    plt.legend()
else:
    raise ValueError("Invalid Nr value")    

plt.grid(alpha=0.3)
plt.show()